In [1]:
# ============================================================
# FIX FAISS / OPENMP CRASH
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(
    "OpenMP Fix Applied"
)

OpenMP Fix Applied


In [2]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import time
import pickle
import psutil

import pandas as pd

from dotenv import load_dotenv

from rank_bm25 import BM25Okapi

from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama

/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_33960/2978333111.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

print(
    "Environment Variables Loaded"
)

Environment Variables Loaded


In [4]:
# ============================================================
# LOAD OPENAI EMBEDDINGS
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print(
    "Embeddings Loaded Successfully"
)

Embeddings Loaded Successfully


In [5]:
# ============================================================
# LOAD FAISS VECTOR STORE
# ============================================================

vector_store = FAISS.load_local(

    "faiss_20k",

    embeddings,

    allow_dangerous_deserialization=True

)

print(
    "FAISS Loaded Successfully"
)

FAISS Loaded Successfully


In [6]:
# ============================================================
# LOAD BM25 INDEX
# ============================================================

with open(
    "bm25_20k.pkl",
    "rb"
) as f:

    bm25 = pickle.load(f)

print(
    "BM25 Loaded Successfully"
)

BM25 Loaded Successfully


In [7]:
# ============================================================
# LOAD ALL CHUNKS
# ============================================================

with open(
    "all_chunks.pkl",
    "rb"
) as f:

    all_chunks = pickle.load(f)

print(
    "Chunks Loaded:",
    len(all_chunks)
)

Chunks Loaded: 160571


In [8]:
# ============================================================
# LOAD LLAMA3 MODEL
# ============================================================

llm = ChatOllama(

    model="llama3",

    temperature=0

)

print(
    "Llama3 Loaded Successfully"
)

Llama3 Loaded Successfully


In [9]:
# ============================================================
# TEST LLAMA3
# ============================================================

response = llm.invoke(
    "Say Hello"
)

print(
    response.content
)

Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?


In [10]:
# ============================================================
# EVALUATION QUERY SET
# ============================================================

queries = [

    "Harry Potter actor",

    "World Cup winner",

    "US election",

    "climate change",

    "Olympics",

    "stock market",

    "global warming",

    "terrorism",

    "education reforms",

    "covid vaccine",

    "space exploration",

    "economic crisis",

    "oil prices",

    "artificial intelligence",

    "healthcare policy",

    "renewable energy",

    "sports championship",

    "government budget",

    "scientific discovery",

    "international relations"

]

print(
    "Queries Loaded:",
    len(queries)
)

Queries Loaded: 20


In [11]:
results = []

In [12]:
# ============================================================
# RECIPROCAL RANK FUSION
# ============================================================

def reciprocal_rank_fusion(
    dense_docs,
    sparse_docs,
    k=60
):

    scores = {}

    for rank, doc in enumerate(
        dense_docs
    ):

        text = doc.page_content

        scores[text] = (

            scores.get(
                text,
                0
            )

            +

            1 / (k + rank + 1)

        )

    for rank, text in enumerate(
        sparse_docs
    ):

        scores[text] = (

            scores.get(
                text,
                0
            )

            +

            1 / (k + rank + 1)

        )

    ranked = sorted(

        scores.items(),

        key=lambda x: x[1],

        reverse=True

    )

    return ranked

In [13]:
# ============================================================
# HYBRID RAG PIPELINE
# ============================================================

for query in queries:

    print(
        f"\nProcessing Query: {query}"
    )

    start_time = time.time()

    memory_before = (

        psutil.Process()
        .memory_info()
        .rss

        /

        1024

        /

        1024

    )

    # ========================================================
    # DENSE RETRIEVAL
    # ========================================================

    dense_docs = vector_store.similarity_search(

        query,

        k=10

    )

    print(
        "Dense Results:",
        len(dense_docs)
    )

    # ========================================================
    # SPARSE RETRIEVAL
    # ========================================================

    tokenized_query = query.lower().split()

    sparse_indices = bm25.get_top_n(

        tokenized_query,

        list(
            range(
                len(all_chunks)
            )
        ),

        n=10

    )

    sparse_docs = [

        all_chunks[idx]

        for idx in sparse_indices
    ]

    print(
        "Sparse Results:",
        len(sparse_docs)
    )

    # ========================================================
    # RRF
    # ========================================================

    fused_docs = reciprocal_rank_fusion(

        dense_docs,

        sparse_docs

    )

    top_docs = [

        doc[0]

        for doc in fused_docs[:5]

    ]

    print(
        "Fusion Documents:",
        len(top_docs)
    )

    # ========================================================
    # BUILD CONTEXT
    # ========================================================

    context = "\n\n".join(
        top_docs
    )

    print(
        "Context Length:",
        len(context)
    )

    # ========================================================
    # CREATE PROMPT
    # ========================================================

    prompt = f"""

    You are a helpful AI assistant.

    Use ONLY the provided context.

    Context:

    {context}

    Question:

    {query}

    Answer:

    """

    # ========================================================
    # GENERATE RESPONSE USING LLAMA3
    # ========================================================

    response = llm.invoke(
        prompt
    )

    answer = response.content

    # ========================================================
    # LATENCY
    # ========================================================

    latency = (

        time.time()

        -

        start_time

    )

    # ========================================================
    # MEMORY
    # ========================================================

    memory_after = (

        psutil.Process()
        .memory_info()
        .rss

        /

        1024

        /

        1024

    )

    memory_used = (

        memory_after

        -

        memory_before

    )

    # ========================================================
    # STORE RESULTS
    # ========================================================

    results.append({

        "pipeline":
        "Hybrid Llama3",

        "retrieval_method":
        "Hybrid",

        "model":
        "Llama3",

        "query":
        query,

        "retrieved_count":
        len(top_docs),

        "retrieved_docs":
        str(top_docs),

        "context":
        context,

        "context_length":
        len(context),

        "generated_answer":
        answer,

        "response_length":
        len(answer),

        "latency_seconds":
        latency,

        "memory_mb":
        memory_used

    })

print(
    "\nHybrid Llama3 Pipeline Completed"
)


Processing Query: Harry Potter actor
Dense Results: 10
Sparse Results: 10
Fusion Documents: 5
Context Length: 2488

Processing Query: World Cup winner
Dense Results: 10
Sparse Results: 10
Fusion Documents: 5
Context Length: 1845

Processing Query: US election
Dense Results: 10
Sparse Results: 10
Fusion Documents: 5
Context Length: 2486

Processing Query: climate change
Dense Results: 10
Sparse Results: 10
Fusion Documents: 5
Context Length: 2052

Processing Query: Olympics
Dense Results: 10
Sparse Results: 10
Fusion Documents: 5
Context Length: 2052

Processing Query: stock market
Dense Results: 10
Sparse Results: 10
Fusion Documents: 5
Context Length: 2156

Processing Query: global warming
Dense Results: 10
Sparse Results: 10
Fusion Documents: 5
Context Length: 2489

Processing Query: terrorism
Dense Results: 10
Sparse Results: 10
Fusion Documents: 5
Context Length: 2299

Processing Query: education reforms
Dense Results: 10
Sparse Results: 10
Fusion Documents: 5
Context Length: 2490

In [14]:
# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

print(
    "Total Records:",
    len(results_df)
)

results_df.head()

Total Records: 20


,pipeline,retrieval_method,model,query,retrieved_count,retrieved_docs,context,context_length,generated_answer,response_length,latency_seconds,memory_mb
0,Hybrid Llama3,Hybrid,Llama3,Harry Potter actor,5,['halfblood prince due for international relea...,halfblood prince due for international release...,2488,"Based on the provided context, I would answer:...",80,4.483082,1092.000000
1,Hybrid Llama3,Hybrid,Llama3,World Cup winner,5,['mental floss although portuguese striker eus...,mental floss although portuguese striker euseb...,1845,"According to the provided context, the answer ...",72,2.889841,4.890625
2,Hybrid Llama3,Hybrid,Llama3,US election,5,['their intentions and not raise hopes only to...,their intentions and not raise hopes only to d...,2486,"Based on the provided context, it appears that...",634,16.086214,0.609375
3,Hybrid Llama3,Hybrid,Llama3,climate change,5,['climate change may be greater than the effec...,climate change may be greater than the effect ...,2052,"Based on the provided context, here's a helpfu...",1192,32.066486,-1773.125000
4,Hybrid Llama3,Hybrid,Llama3,Olympics,5,['one another thats what the olympics are abou...,one another thats what the olympics are about ...,2052,"According to the provided context, the Olympic...",597,18.153795,191.578125


In [15]:
# ============================================================
# CHECK DATAFRAME STRUCTURE
# ============================================================

results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   pipeline          20 non-null     str    
 1   retrieval_method  20 non-null     str    
 2   model             20 non-null     str    
 3   query             20 non-null     str    
 4   retrieved_count   20 non-null     int64  
 5   retrieved_docs    20 non-null     str    
 6   context           20 non-null     str    
 7   context_length    20 non-null     int64  
 8   generated_answer  20 non-null     str    
 9   response_length   20 non-null     int64  
 10  latency_seconds   20 non-null     float64
 11  memory_mb         20 non-null     float64
dtypes: float64(2), int64(3), str(7)
memory usage: 106.1 KB


In [16]:
# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(

    "hybrid_llama3_results.csv",

    index=False

)

print(
    "Hybrid Llama3 Results Saved Successfully"
)

Hybrid Llama3 Results Saved Successfully
